# $t_b$ における $\tau_1^\theta$ の独立検証

In [1]:
from sage.all import FreeGroup, ZZ, QQ, vector, matrix

%display latex

F = FreeGroup(4, 'x,y,z,w')
x, y, z, w = F.generators()
BASIS = (x, y, z, w)

def comm(u, v):
    return u * v * (~u) * (~v)

bnd = comm(x, y) * comm(z, w)

X_H = vector(QQ, [1, 0, 0, 0])
Y_H = vector(QQ, [0, 1, 0, 0])
Z_H = vector(QQ, [0, 0, 1, 0])
W_H = vector(QQ, [0, 0, 0, 1])
H_STANDARD_BASIS = (X_H, Y_H, Z_H, W_H)

J = matrix(QQ, [
    [ 0,  1,  0, 0],
    [-1,  0,  0, 0],
    [ 0,  0,  0, 1],
    [ 0,  0, -1, 0],
])

WEDGE_PAIRS = (
    (0, 1), (0, 2), (0, 3),
    (1, 2), (1, 3), (2, 3)
)
TRIPLE_PAIRS = ((0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3))

def matrix_from_columns(columns, ring=QQ):
    return matrix(ring, len(columns[0]), len(columns),
                  lambda i, j: columns[j][i])

def wedge(u, v):
    u = vector(QQ, u)
    v = vector(QQ, v)
    return vector(QQ, [u[i]*v[j] - u[j]*v[i]
                       for i, j in WEDGE_PAIRS])

## Route A — Dehn twist formula

In [2]:
# b = y^(-1) z w z^(-1)
# |b| = -Y + W
# ell2(b) = 1/2 X∧Y - 1/2 Y∧W + 1/2 Z∧W

b_H_A = -Y_H + W_H
ell_b_A = (
      QQ(1)/2 * wedge(X_H, Y_H)
    - QQ(1)/2 * wedge(Y_H, W_H)
    + QQ(1)/2 * wedge(Z_H, W_H)
)

def eta_coeff(eta, i, j):
    if i == j:
        return QQ(0)
    if i < j:
        return eta[WEDGE_PAIRS.index((i, j))]
    return -eta[WEDGE_PAIRS.index((j, i))]

def wedge_H_Lambda2(u, eta):
    u = vector(QQ, u)
    eta = vector(QQ, eta)
    return vector(QQ, [
        u[i] * eta_coeff(eta, j, k)
        - u[j] * eta_coeff(eta, i, k)
        + u[k] * eta_coeff(eta, i, j)
        for i, j, k in TRIPLE_PAIRS
    ])

def symplectic_pair(u, v):
    return vector(QQ, u).dot_product(J * vector(QQ, v))

def lambda3_to_hom(lam):
    columns = []
    for h in H_STANDARD_BASIS:
        eta = vector(QQ, 6)
        for coeff, (i, j, k) in zip(lam, TRIPLE_PAIRS):
            e_i = H_STANDARD_BASIS[i]
            e_j = H_STANDARD_BASIS[j]
            e_k = H_STANDARD_BASIS[k]
            eta += coeff * (
                symplectic_pair(h, e_i) * wedge(e_j, e_k)
                - symplectic_pair(h, e_j) * wedge(e_i, e_k)
                + symplectic_pair(h, e_k) * wedge(e_i, e_j)
            )
        columns.append(eta)
    return matrix_from_columns(columns, QQ)

lambda_route_A = -wedge_H_Lambda2(b_H_A, ell_b_A)
tau_route_A = lambda3_to_hom(lambda_route_A)

print('|b| =', b_H_A)
print('ell2(b) =', ell_b_A)
print('-|b| wedge ell2(b) =', lambda_route_A)
print('tau_1^theta(t_b), Route A =')
print(tau_route_A)

|b| = (0, -1, 0, 1)
ell2(b) = (1/2, 0, 0, 0, -1/2, 1/2)
-|b| wedge ell2(b) = (0, -1/2, 0, 1/2)
tau_1^theta(t_b), Route A =
[   0    0 -1/2    0]
[   0    0    0    0]
[ 1/2    0    0    0]
[   0    0  1/2    0]
[   0  1/2    0  1/2]
[ 1/2    0    0    0]


## Route B — explicit calculation on $\pi$

In [3]:
# 4本の生成元像を自由群の語として直接入力
tb_x = x * (~y) * z * w * (~z)
tb_y = z * (~w) * (~z) * y * z * w * (~z)
tb_z = z * (~w) * (~z) * y * z
tb_w = w
images_b = (tb_x, tb_y, tb_z, tb_w)

for name, image in zip(('x', 'y', 'z', 'w'), images_b):
    print(f't_b({name}) = {image}')

t_b(x) = x*y^-1*z*w*z^-1
t_b(y) = z*w^-1*z^-1*y*z*w*z^-1
t_b(z) = z*w^-1*z^-1*y*z
t_b(w) = w


In [4]:
# 各語の指数和を直接数える
def homology(element):
    v = vector(ZZ, 4)
    for idx in element.Tietze():
        i = abs(idx) - 1
        v[i] += 1 if idx > 0 else -1
    return v

H_tb_x = homology(tb_x)
H_tb_y = homology(tb_y)
H_tb_z = homology(tb_z)
H_tb_w = homology(tb_w)

print('|t_b(x)| =', H_tb_x)
print('|t_b(y)| =', H_tb_y)
print('|t_b(z)| =', H_tb_z)
print('|t_b(w)| =', H_tb_w)

A_route_B = matrix_from_columns(
    [H_tb_x, H_tb_y, H_tb_z, H_tb_w], ZZ
)

print('A, Route B =')
print(A_route_B)
assert A_route_B.transpose() * J * A_route_B == J

|t_b(x)| = (1, -1, 0, 1)
|t_b(y)| = (0, 1, 0, 0)
|t_b(z)| = (0, 1, 1, -1)
|t_b(w)| = (0, 0, 0, 1)
A, Route B =
[ 1  0  0  0]
[-1  1  1  0]
[ 0  0  1  0]
[ 1  0 -1  1]


In [5]:
# ell2 を語の左から一文字ずつ計算
ELL2_GENERATORS = {
    1: vector(QQ, [ QQ(1)/2, 0, 0, 0, 0, 0]),
    2: vector(QQ, [-QQ(1)/2, 0, 0, 0, 0, 0]),
    3: vector(QQ, [0, 0, 0, 0, 0,  QQ(1)/2]),
    4: vector(QQ, [0, 0, 0, 0, 0, -QQ(1)/2]),
}

def ell2_letter(idx):
    value = vector(QQ, ELL2_GENERATORS[abs(idx)])
    return value if idx > 0 else -value

def ell2_word(element):
    prefix_H = vector(QQ, 4)
    result = vector(QQ, 6)
    for idx in element.Tietze():
        letter_H = vector(QQ, 4)
        i = abs(idx) - 1
        letter_H[i] = 1 if idx > 0 else -1
        result += (
            ell2_letter(idx)
            + QQ(1)/2 * wedge(prefix_H, letter_H)
        )
        prefix_H += letter_H
    return result

ell_tb_x = ell2_word(tb_x)
ell_tb_y = ell2_word(tb_y)
ell_tb_z = ell2_word(tb_z)
ell_tb_w = ell2_word(tb_w)

print('ell2(t_b(x)) =', ell_tb_x)
print('ell2(t_b(y)) =', ell_tb_y)
print('ell2(t_b(z)) =', ell_tb_z)
print('ell2(t_b(w)) =', ell_tb_w)
assert ell2_word(bnd) == vector(QQ, [1, 0, 0, 0, 0, 1])

ell2(t_b(x)) = (1/2, 0, 1/2, 0, -1/2, 1/2)
ell2(t_b(y)) = (-1/2, 0, 0, 0, 1, 0)
ell2(t_b(z)) = (-1/2, 0, 0, 1/2, 1/2, 1/2)
ell2(t_b(w)) = (0, 0, 0, 0, 0, -1/2)


In [6]:
def wedge_action_matrix(A):
    A = A.change_ring(QQ)
    return matrix_from_columns([
        wedge(A.column(i), A.column(j))
        for i, j in WEDGE_PAIRS
    ], QQ)

A2_route_B = wedge_action_matrix(A_route_B)

A2_ell_x = A2_route_B * ell2_word(x)
A2_ell_y = A2_route_B * ell2_word(y)
A2_ell_z = A2_route_B * ell2_word(z)
A2_ell_w = A2_route_B * ell2_word(w)

delta_x = ell_tb_x - A2_ell_x
delta_y = ell_tb_y - A2_ell_y
delta_z = ell_tb_z - A2_ell_z
delta_w = ell_tb_w - A2_ell_w

print('Lambda^2 A =')
print(A2_route_B)
print('Delta_x =', delta_x)
print('Delta_y =', delta_y)
print('Delta_z =', delta_z)
print('Delta_w =', delta_w)

D_route_B = matrix_from_columns(
    [delta_x, delta_y, delta_z, delta_w], QQ
)
A_route_B_inverse = A_route_B.change_ring(QQ).inverse()
tau_route_B = D_route_B * A_route_B_inverse

print('D, Route B =')
print(D_route_B)
print('A^(-1), Route B =')
print(A_route_B_inverse)
print('tau_1^theta(t_b), Route B =')
print(tau_route_B)

Lambda^2 A =
[ 1  1  0  0  0  0]
[ 0  1  0  0  0  0]
[ 0 -1  1  0  0  0]
[ 0 -1  0  1  0  0]
[-1  0 -1 -1  1  1]
[ 0 -1  0  0  0  1]
Delta_x = (0, 0, 1/2, 0, 0, 1/2)
Delta_y = (0, 0, 0, 0, 1/2, 0)
Delta_z = (-1/2, 0, 0, 1/2, 0, 0)
Delta_w = (0, 0, 0, 0, 1/2, 0)
D, Route B =
[   0    0 -1/2    0]
[   0    0    0    0]
[ 1/2    0    0    0]
[   0    0  1/2    0]
[   0  1/2    0  1/2]
[ 1/2    0    0    0]
A^(-1), Route B =
[ 1  0  0  0]
[ 1  1 -1  0]
[ 0  0  1  0]
[-1  0  1  1]
tau_1^theta(t_b), Route B =
[   0    0 -1/2    0]
[   0    0    0    0]
[ 1/2    0    0    0]
[   0    0  1/2    0]
[   0  1/2    0  1/2]
[ 1/2    0    0    0]


## Comparison

In [7]:
print('Route A =')
print(tau_route_A)
print('Route B =')
print(tau_route_B)
print('Route A == Route B :', tau_route_A == tau_route_B)

assert tau_route_A == tau_route_B

Route A =
[   0    0 -1/2    0]
[   0    0    0    0]
[ 1/2    0    0    0]
[   0    0  1/2    0]
[   0  1/2    0  1/2]
[ 1/2    0    0    0]
Route B =
[   0    0 -1/2    0]
[   0    0    0    0]
[ 1/2    0    0    0]
[   0    0  1/2    0]
[   0  1/2    0  1/2]
[ 1/2    0    0    0]
Route A == Route B : True
